## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([784, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(x @ self.W1 + self.b1)
        logits = h1 @ self.W2 + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

11490434/11490434 [==============================] - 5s 0us/step
epoch 0 : loss 2.5960338 ; accuracy 0.06525
epoch 1 : loss 2.5728455 ; accuracy 0.062816665
epoch 2 : loss 2.5519593 ; accuracy 0.061683334
epoch 3 : loss 2.5329344 ; accuracy 0.061616667
epoch 4 : loss 2.5154376 ; accuracy 0.06251667
epoch 5 : loss 2.4992135 ; accuracy 0.06401667
epoch 6 : loss 2.4840598 ; accuracy 0.06566667
epoch 7 : loss 2.4698117 ; accuracy 0.06865
epoch 8 : loss 2.4563434 ; accuracy 0.0719
epoch 9 : loss 2.4435475 ; accuracy 0.07551666
epoch 10 : loss 2.4313369 ; accuracy 0.079616666
epoch 11 : loss 2.4196408 ; accuracy 0.08438333
epoch 12 : loss 2.408401 ; accuracy 0.08928333
epoch 13 : loss 2.3975677 ; accuracy 0.094066665
epoch 14 : loss 2.3870978 ; accuracy 0.09928333
epoch 15 : loss 2.3769524 ; accuracy 0.104616664
epoch 16 : loss 2.3671005 ; accuracy 0.1097
epoch 17 : loss 2.3575141 ; accuracy 0.1153
epoch 18 : loss 2.3481681 ; accuracy 0.12036667
epoch 19 : loss 2.3390405 ; accuracy 0.1258166